## 1. Environment Setup

To host a web application from within a Google Colab environment, we use `cloudflared` to create a secure public tunnel that points to the Streamlit server running on this virtual machine.

In [1]:
# --- Install Dependencies ---
!pip install streamlit -q

# Download Cloudflare Tunnel binary
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 35.0 MB/s eta 0:00:00
--2026-05-06 00:36:57--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-05-06 00:36:57--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-06T01%3A12%3A58Z&rscd=attachment%3B+f

## 2. Developing the Application Script

In Streamlit development, we typically work with a standalone `.py` file. In Colab, we use the `%%writefile` magic command to save our code into `app.py`.

**Note on the Execution Model:** Every time a user interacts with a widget, Streamlit reruns your entire Python script from top to bottom. This is a fundamental concept that ensures the UI always matches the current state of your variables.

In [2]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from datasets import load_dataset

# ---------------------------------------------------------
# Page Configuration
# ---------------------------------------------------------
st.set_page_config(
    page_title="Netflix Mobile App Experience Dashboard",
    page_icon="📱",
    layout="wide"
)

# ---------------------------------------------------------
# Data Loading and Preprocessing
# ---------------------------------------------------------
@st.cache_data
def load_and_prepare_data():
    # Load Netflix app review dataset from Hugging Face
    dataset = load_dataset("splevine/netflix-app-review-sample")
    df = dataset["netflix"].to_pandas()

    # Keep original row count for dashboard context
    original_rows = df.shape[0]

    # Remove exact duplicate rows
    df_clean = df.drop_duplicates().copy()

    # Remove rows with missing review text or rating
    df_clean = df_clean.dropna(subset=["content", "rating"])

    # Convert review text and rating to appropriate formats
    df_clean["content"] = df_clean["content"].astype(str)
    df_clean["rating"] = pd.to_numeric(df_clean["rating"], errors="coerce")
    df_clean = df_clean.dropna(subset=["rating"])

    # Clean review text for NLP/dashboard search
    def clean_text(text):
        text = text.lower()
        text = re.sub(r"http\S+|www\S+", "", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    df_clean["clean_review"] = df_clean["content"].apply(clean_text)

    # Create modeling dataset by removing neutral 3-star reviews
    df_model = df_clean[df_clean["rating"] != 3].copy()

    # Create sentiment label
    # 1 = Negative, 0 = Non-negative
    df_model["negative_sentiment"] = (df_model["rating"] <= 2).astype(int)

    # Add readable sentiment label for dashboard use
    df_model["sentiment_label"] = df_model["negative_sentiment"].map({
        1: "Negative",
        0: "Non-negative"
    })

    # Add review length for optional EDA/dashboard filtering
    df_model["review_length"] = df_model["clean_review"].apply(lambda x: len(x.split()))

    cleaning_summary = {
        "original_rows": original_rows,
        "cleaned_rows": df_clean.shape[0],
        "modeled_rows": df_model.shape[0],
        "duplicates_removed": original_rows - df_clean.shape[0],
        "neutral_reviews_removed": df_clean.shape[0] - df_model.shape[0],
        "negative_reviews": int(df_model["negative_sentiment"].sum()),
        "nonnegative_reviews": int((df_model["negative_sentiment"] == 0).sum()),
        "negative_share": df_model["negative_sentiment"].mean()
    }

    return df_clean, df_model, cleaning_summary


df_clean, df_model, cleaning_summary = load_and_prepare_data()

# ---------------------------------------------------------
# Helper Function for Readable Charts
# ---------------------------------------------------------
def make_bar_chart(data, x_col, y_col, title, xlabel, ylabel, rotation=30):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(data[x_col], data[y_col])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotation)
    plt.tight_layout()
    return fig

# ---------------------------------------------------------
# Dashboard Title and Executive Framing
# ---------------------------------------------------------
st.title("📱 Netflix Mobile App Experience Executive Dashboard")

st.markdown("""
This dashboard translates Netflix mobile app review analysis into executive-level insights.
It is designed to help leadership quickly understand customer sentiment, identify major
drivers of dissatisfaction, and assess how NLP and machine learning can support product
decision-making.
""")

st.divider()

# ---------------------------------------------------------
# Executive KPI Section
# ---------------------------------------------------------
st.header("Executive Summary KPIs")

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        label="Modeled Reviews",
        value=f"{cleaning_summary['modeled_rows']:,}",
        help="Reviews used for sentiment classification after removing duplicates and neutral 3-star reviews."
    )

with col2:
    st.metric(
        label="Negative Sentiment Share",
        value=f"{cleaning_summary['negative_share']:.1%}",
        help="Share of modeled reviews classified as negative based on rating labels."
    )

with col3:
    st.metric(
        label="Negative Recall",
        value="87.61%",
        help="Share of actual negative reviews correctly detected by the Logistic Regression model."
    )

with col4:
    st.metric(
        label="Best F1-Score",
        value="78.82%",
        help="Best F1-score from model comparison, achieved by Random Forest."
    )

st.markdown("""
**Executive takeaway:** Negative app feedback is substantial and measurable. The dashboard
focuses on the customer-experience issues most relevant to Netflix leadership: viewing friction,
account access, app usability, and user interface concerns.
""")

# ---------------------------------------------------------
# Data Integration Summary
# ---------------------------------------------------------
with st.expander("View data integration and cleaning summary"):
    st.write("This dashboard uses the Netflix mobile app review dataset from Assignment 2.")
    st.write(f"Original rows loaded: {cleaning_summary['original_rows']:,}")
    st.write(f"Duplicate rows removed: {cleaning_summary['duplicates_removed']:,}")
    st.write(f"Cleaned unique reviews: {cleaning_summary['cleaned_rows']:,}")
    st.write(f"Neutral 3-star reviews removed: {cleaning_summary['neutral_reviews_removed']:,}")
    st.write(f"Final modeled reviews: {cleaning_summary['modeled_rows']:,}")
    st.write(f"Negative reviews: {cleaning_summary['negative_reviews']:,}")
    st.write(f"Non-negative reviews: {cleaning_summary['nonnegative_reviews']:,}")

    st.dataframe(
        df_model[["rating", "sentiment_label", "content", "clean_review"]].head(10),
        use_container_width=True
    )

st.divider()

# ---------------------------------------------------------
# Sidebar Controls
# ---------------------------------------------------------
st.sidebar.title("Dashboard Controls")

st.sidebar.markdown("""
Use these controls to explore customer sentiment, review themes, and model outputs.
""")

sentiment_filter = st.sidebar.selectbox(
    "Select sentiment view",
    ["All Reviews", "Negative Reviews", "Non-Negative Reviews"]
)

rating_filter = st.sidebar.multiselect(
    "Select rating(s)",
    options=sorted(df_model["rating"].unique()),
    default=sorted(df_model["rating"].unique())
)

keyword_filter = st.sidebar.text_input(
    "Search review keyword",
    placeholder="Example: account, watch, app, ui"
)

st.sidebar.markdown("---")
st.sidebar.caption("Designed for executive exploration, not technical model debugging.")

# ---------------------------------------------------------
# Apply Sidebar Filters
# ---------------------------------------------------------
filtered_df = df_model.copy()

if sentiment_filter == "Negative Reviews":
    filtered_df = filtered_df[filtered_df["sentiment_label"] == "Negative"]
elif sentiment_filter == "Non-Negative Reviews":
    filtered_df = filtered_df[filtered_df["sentiment_label"] == "Non-negative"]

if rating_filter:
    filtered_df = filtered_df[filtered_df["rating"].isin(rating_filter)]

if keyword_filter.strip() != "":
    keyword = keyword_filter.lower().strip()
    filtered_df = filtered_df[
        filtered_df["clean_review"].str.contains(keyword, na=False)
    ]

# ---------------------------------------------------------
# Main Dashboard Sections
# ---------------------------------------------------------
st.header("1. Customer Sentiment Overview")

st.markdown("""
This section summarizes the overall review sentiment distribution and rating pattern.
For executives, the goal is to quickly understand the scale of negative app feedback.
""")

st.write(f"Reviews currently displayed after filters: **{len(filtered_df):,}**")

col_sentiment, col_rating = st.columns(2)

with col_sentiment:
    sentiment_counts = (
        filtered_df["sentiment_label"]
        .value_counts()
        .reset_index()
    )
    sentiment_counts.columns = ["Sentiment", "Number of Reviews"]

    if sentiment_counts.empty:
        st.warning("No reviews match the selected filters.")
    else:
        fig1 = make_bar_chart(
            sentiment_counts,
            "Sentiment",
            "Number of Reviews",
            "Sentiment Distribution",
            "Sentiment",
            "Number of Reviews",
            rotation=0
        )
        st.pyplot(fig1)

with col_rating:
    rating_counts = (
        filtered_df["rating"]
        .value_counts()
        .sort_index()
        .reset_index()
    )
    rating_counts.columns = ["Rating", "Number of Reviews"]

    if rating_counts.empty:
        st.warning("No ratings available for the selected filters.")
    else:
        fig2 = make_bar_chart(
            rating_counts,
            "Rating",
            "Number of Reviews",
            "Rating Distribution",
            "Rating",
            "Number of Reviews",
            rotation=0
        )
        st.pyplot(fig2)

st.markdown("""
**Executive interpretation:** Negative sentiment represents a meaningful share of Netflix
mobile app feedback. The rating and sentiment views help leadership understand whether
customer dissatisfaction is concentrated in low ratings or spread more broadly across reviews.
""")

st.divider()

# ---------------------------------------------------------
# Key Dissatisfaction Drivers
# ---------------------------------------------------------
st.header("2. Key Dissatisfaction Drivers")

st.markdown("""
This section highlights the most frequent words and two-word phrases found in negative
reviews. These charts help identify recurring sources of app-related dissatisfaction.
""")

# Assignment 2 NLP results
negative_word_freq = pd.DataFrame({
    "Term": ["netflix", "app", "watch", "account"],
    "Mentions": [409, 309, 184, 123]
})

negative_bigram_freq = pd.DataFrame({
    "Phrase": ["don want", "want watch", "worst app", "new ui"],
    "Mentions": [21, 20, 16, 16]
})

col_words, col_phrases = st.columns(2)

with col_words:
    fig3 = make_bar_chart(
        negative_word_freq,
        "Term",
        "Mentions",
        "Top Words in Negative Reviews",
        "Word",
        "Number of Mentions",
        rotation=0
    )
    st.pyplot(fig3)

with col_phrases:
    fig4 = make_bar_chart(
        negative_bigram_freq,
        "Phrase",
        "Mentions",
        "Top Two-Word Phrases in Negative Reviews",
        "Phrase",
        "Number of Mentions",
        rotation=20
    )
    st.pyplot(fig4)

st.markdown("""
**Executive interpretation:** The most common negative-review terms point to app usage,
watching friction, and account-related problems. The bigram results are especially useful
because phrases such as **"want watch," "worst app,"** and **"new ui"** suggest specific
experience issues that leadership can connect to product and design priorities.
""")

st.divider()

# ---------------------------------------------------------
# Filtered Review Explorer
# ---------------------------------------------------------
st.header("3. Review Explorer")

st.markdown("""
This section allows executives to inspect examples of reviews behind the dashboard metrics.
The table updates based on the sidebar sentiment, rating, and keyword filters.
""")

review_display = filtered_df[[
    "rating",
    "sentiment_label",
    "content",
    "review_length"
]].copy()

review_display = review_display.rename(columns={
    "rating": "Rating",
    "sentiment_label": "Sentiment",
    "content": "Review Text",
    "review_length": "Review Length"
})

st.dataframe(
    review_display.head(25),
    use_container_width=True
)

st.caption("Showing up to 25 reviews based on the selected filters.")

st.divider()

# ---------------------------------------------------------
# Model Results and Review Monitoring
# ---------------------------------------------------------
st.header("4. Model Results and Review Monitoring")

st.markdown("""
This section communicates model results in an executive-friendly way. The goal is not to
show technical model details, but to show whether NLP can help Netflix detect negative
app experiences at scale.
""")

model_metrics = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "Decision Tree"],
    "Accuracy": [0.8025, 0.8330, 0.7666],
    "Precision": [0.7071, 0.8122, 0.7330],
    "Recall": [0.8761, 0.7655, 0.6681],
    "F1-score": [0.7826, 0.7882, 0.6991]
})

st.dataframe(
    model_metrics,
    use_container_width=True
)

fig5, ax = plt.subplots(figsize=(8, 4))
ax.bar(model_metrics["Model"], model_metrics["F1-score"])
ax.set_title("Classification Model Comparison by F1-Score")
ax.set_xlabel("Model")
ax.set_ylabel("F1-Score")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
st.pyplot(fig5)

st.markdown("""
**Executive interpretation:** Random Forest produced the strongest overall F1-score,
while Logistic Regression had the strongest recall for negative sentiment. For Netflix,
high recall is valuable because it means fewer dissatisfied users are missed. This supports
using NLP sentiment monitoring as an early-warning system for app-experience problems.
""")

st.divider()

# ---------------------------------------------------------
# Executive Recommendations
# ---------------------------------------------------------
st.header("5. Executive Recommendations")

st.markdown("""
Based on the dashboard findings, Netflix leadership should consider the following actions:

1. **Monitor negative sentiment after app updates**, especially when phrases such as
   "new ui" or "worst app" increase.
2. **Prioritize viewing friction and app reliability**, because negative reviews frequently
   reference app usage and watching-related issues.
3. **Investigate account-related complaints**, since "account" is one of the most frequent
   words in negative reviews.
4. **Use NLP-based review monitoring as an early-warning system** so product and customer
   experience teams can respond faster to emerging dissatisfaction patterns.

**Bottom line:** The dashboard turns app reviews into actionable business intelligence by
showing how much dissatisfaction exists, what users are complaining about, and how machine
learning can help Netflix detect negative experiences at scale.
""")

Writing app.py


## 3. Running the App and Tunneling

Run the cell below to start the Streamlit server and generate your public URL.

**Instructions:**
1. Run the cell.
2. Look for the line starting with `https://...trycloudflare.com`.
3. Click that link to view your live dashboard.

**Note:** Wait for 10-20 seconds before clicking on the link, to give enough time for the URL to be generated.

In [3]:
import os

print("🚀 Starting Streamlit and Cloudflare Tunnel...")
print("👉 Open the Cloudflare URL shown below")

# --- Run Streamlit App ---
# We run streamlit in the background (&)
os.system("streamlit run app.py & >/dev/null 2>&1")

# --- Expose with Cloudflare Tunnel ---
# This will generate a public .trycloudflare.com URL
!cloudflared tunnel --url http://localhost:8501

🚀 Starting Streamlit and Cloudflare Tunnel...
👉 Open the Cloudflare URL shown below
2026-05-06T00:37:01Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-06T00:37:01Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-06T00:37:04Z INF +--------------------------------------------------------------------------------------------+
2026-05-06T00:37:04Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable): 